<a href="https://colab.research.google.com/github/jaywong8/Detection-of-Atrial-Fibrillation-Termination/blob/main/Parsing_Positive_Segments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

installing required libraries

In [1]:
!pip -q install wfdb pandas numpy scipy scikit-learn matplotlib

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import pandas as pd
from pathlib import Path

# Existing project files in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/icentia11k_afib_audit")
PILOT_DIR = PROJECT_DIR / "pilot_10s_30s"

CANDIDATE_FILE = PILOT_DIR / "candidate_window_index.csv"
MANIFEST_FILE = PILOT_DIR / "patient_split_manifest.csv"

OUTPUT_FILE = PILOT_DIR / "train_windows_patient_episode_capped_natural.csv"
AUDIT_FILE = PILOT_DIR / "train_sampling_audit.csv"

SEED = 42
MAX_PER_EPISODE = 3
MAX_PER_PATIENT = 100

# 1. Verify source files
for path in [CANDIDATE_FILE, MANIFEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Check the existing project directory before proceeding."
        )

windows = pd.read_csv(CANDIDATE_FILE)
manifest = pd.read_csv(MANIFEST_FILE)

# 2. Verify expected schemas
required_windows = {
    "patient", "record_stem", "episode_id", "label", "partition"
}
required_manifest = {"patient", "partition"}

assert required_windows.issubset(windows.columns), (
    f"Unexpected candidate-index columns: {windows.columns.tolist()}"
)
assert required_manifest.issubset(manifest.columns), (
    f"Unexpected manifest columns: {manifest.columns.tolist()}"
)

# 3. Verify patient-level split consistency
windows["patient"] = windows["patient"].astype(str)
manifest["patient"] = manifest["patient"].astype(str)

assert not manifest["patient"].duplicated().any(), (
    "Duplicate patient assignments in split manifest."
)

windows["partition"] = windows["partition"].astype(str).str.lower().str.strip()
manifest["partition"] = manifest["partition"].astype(str).str.lower().str.strip()

patient_partition = manifest.set_index("patient")["partition"]
expected_partition = windows["patient"].map(patient_partition)

assert expected_partition.notna().all(), (
    "Some candidate-index patients are missing from the split manifest."
)
assert windows["partition"].eq(expected_partition).all(), (
    "Candidate index conflicts with the patient split manifest. "
    "Stop and reconcile before sampling."
)

print("Verified patient split.")
print("Manifest patients:", manifest["patient"].nunique())
print("Candidate-index rows:", len(windows))
print("\nPatients by partition:")
print(manifest.groupby("partition")["patient"].nunique())

# 4. Select training partition only
train = windows.loc[windows["partition"].eq("train")].copy()

assert not train.empty, (
    f"No rows labeled 'train'. Found partitions: "
    f"{sorted(windows['partition'].unique())}"
)

episode_cols = ["patient", "record_stem", "episode_id"]

assert train[episode_cols].notna().all().all(), (
    "Missing episode identifiers; refusing to sample."
)

# 5. Cap at 3 windows per episode
# Stable sorting plus seeded sampling makes the result reproducible.
train = train.sort_values(episode_cols + ["window_id"]) \
    if "window_id" in train.columns else train.sort_values(episode_cols)

episode_sample = (
    train.groupby(episode_cols, sort=True, group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(len(g), MAX_PER_EPISODE),
            random_state=SEED
        )
    )
    .reset_index(drop=True)
)

# 6. Cap at 100 windows per patient
sampled = (
    episode_sample.groupby("patient", sort=True, group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(len(g), MAX_PER_PATIENT),
            random_state=SEED
        )
    )
    .reset_index(drop=True)
)

# 7. Validate before saving
assert sampled["partition"].eq("train").all()
assert sampled.groupby("patient").size().max() <= MAX_PER_PATIENT
assert sampled.groupby(episode_cols).size().max() <= MAX_PER_EPISODE

PILOT_DIR.mkdir(parents=True, exist_ok=True)

sampled.to_csv(OUTPUT_FILE, index=False)

# Save an audit of sampled labels by patient
audit = pd.crosstab(sampled["patient"], sampled["label"])
audit["total_windows"] = audit.sum(axis=1)
audit.to_csv(AUDIT_FILE)

# 8. Report results
print("\nSUCCESS")
print("Training sample:", OUTPUT_FILE)
print("Audit report:", AUDIT_FILE)
print("Sampled rows:", len(sampled))
print("Sampled patients:", sampled["patient"].nunique())
print("\nLabel counts:")
print(sampled["label"].value_counts(dropna=False))
print("\nWindows per patient:")
print(sampled.groupby("patient").size().describe())
print(f"\nCSV size: {OUTPUT_FILE.stat().st_size / 1e6:.2f} MB")

Verified patient split.
Manifest patients: 142
Candidate-index rows: 1463212

Patients by partition:
partition
test          22
train         99
validation    21
Name: patient, dtype: int64


/tmp/ipykernel_594/4202160137.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_594/4202160137.py:103: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



SUCCESS
Training sample: /content/drive/MyDrive/icentia11k_afib_audit/pilot_10s_30s/train_windows_patient_episode_capped_natural.csv
Audit report: /content/drive/MyDrive/icentia11k_afib_audit/pilot_10s_30s/train_sampling_audit.csv
Sampled rows: 8728
Sampled patients: 99

Label counts:
label
1    4760
0    3968
Name: count, dtype: int64

Windows per patient:
count     99.000000
mean      88.161616
std       29.517794
min        1.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
dtype: float64

CSV size: 0.95 MB


Confirming what has been audited

In [ ]:

from pathlib import Path
import pandas as pd

AUDIT_DIR = Path("/content/drive/MyDrive/icentia11k_afib_audit")
AUDIT_CSV = AUDIT_DIR / "audited_segments.csv"

if not AUDIT_CSV.exists():
    raise FileNotFoundError(f"Existing audit not found: {AUDIT_CSV}")

audit = pd.read_csv(AUDIT_CSV)

positive = audit[
    (pd.to_numeric(audit["afib_starts"], errors="coerce").fillna(0) > 0)
    & (audit["status"].astype(str).str.lower() == "ok")
].copy()

print(f"Previously audited segments: {len(audit):,}")
print(f"Previously identified AFib-positive segments: {len(positive):,}")
print(f"Patients represented: {positive['patient_folder'].nunique():,}")
print(f"Existing audit columns: {audit.columns.tolist()}")


Previously audited segments: 99,250
Previously identified AFib-positive segments: 4,538
Patients represented: 141
Existing audit columns: ['patient_folder', 'segment', 'afib_starts', 'nsr_starts', 'afl_starts', 'annotation_count', 'status', 'error']


processing audited segments

In [ ]:

import time
import requests
import pandas as pd
import numpy as np
import wfdb
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================
BASE_URL = "https://physionet.org/files/icentia11k-continuous-ecg/1.0"
AUDIT_DIR = Path("/content/drive/MyDrive/icentia11k_afib_audit")
AUDIT_CSV = AUDIT_DIR / "audited_segments.csv"
CACHE_ROOT = Path("/content/icentia11k_audit")

OUT_CSV = AUDIT_DIR / "afib_episode_audit.csv"
FAIL_CSV = AUDIT_DIR / "afib_episode_audit_failures.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. REUSE ONLY PREVIOUSLY AUDITED AFIB-POSITIVE SEGMENTS
# ============================================================
audit = pd.read_csv(AUDIT_CSV)

positive = audit[
    (pd.to_numeric(audit["afib_starts"], errors="coerce").fillna(0) > 0)
    & (audit["status"].astype(str).str.lower() == "ok")
].copy()

def make_stem(row):
    folder = str(row["patient_folder"]).strip().rstrip("/")
    segment = str(row["segment"]).strip()

    if segment.startswith("p") and "_s" in segment:
        return segment
    if folder.startswith("p") and "_s" in folder:
        return folder

    number = int(segment[1:]) if segment.startswith("s") else int(float(segment))
    return f"{folder}_s{number:02d}"

positive["record_stem"] = positive.apply(make_stem, axis=1)
positive = positive.drop_duplicates("record_stem").reset_index(drop=True)

print(f"Existing AFib-positive segments to process: {len(positive):,}")

# ============================================================
# 2. RETRIEVE MISSING ANNOTATIONS AND PARSE RHYTHM INTERVALS
#    No .dat waveform files are downloaded.
# ============================================================
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
session = requests.Session()
session.headers.update({"User-Agent": "Icentia11k-annotation-audit/1.0"})

results = []
failures = []

def fetch_if_missing(url, path):
    if path.exists() and path.stat().st_size > 0:
        return
    response = session.get(url, timeout=45)
    response.raise_for_status()
    if not response.content:
        raise ValueError(f"Empty response for {url}")
    path.write_bytes(response.content)

def normalize_note(note):
    return str(note).strip()

def rhythm_label(note):
    note = normalize_note(note)
    if note.startswith("("):
        return note[1:].strip().upper()
    return None

for i, row in enumerate(positive.itertuples(index=False), start=1):
    stem = row.record_stem
    patient = stem.split("_")[0]
    group = patient[:3]

    local_dir = CACHE_ROOT / group / patient
    local_dir.mkdir(parents=True, exist_ok=True)
    base = local_dir / stem
    atr_path = Path(str(base) + ".atr")
    hea_path = Path(str(base) + ".hea")

    try:
        for suffix, path in [("atr", atr_path), ("hea", hea_path)]:
            url = f"{BASE_URL}/{group}/{patient}/{stem}.{suffix}"
            fetch_if_missing(url, path)

        ann = wfdb.rdann(str(base), "atr")
        header = wfdb.rdheader(str(base))
        fs = float(header.fs)

        # Keep all annotation notes, not just selected rhythm markers.
        events = []
        for sample, note in zip(ann.sample, ann.aux_note):
            note = normalize_note(note)
            label = rhythm_label(note)
            if label in {"AFIB", "N", "AFL"} or note == ")":
                events.append({
                    "sample": int(sample),
                    "note": note,
                    "label": label,
                })

        # Pair each AFIB start with the first valid subsequent boundary.
        for j, event in enumerate(events):
            if event["label"] != "AFIB":
                continue

            start = event["sample"]
            boundary = None

            for later in events[j + 1:]:
                # Another rhythm-start annotation takes precedence as a
                # boundary; never pair across a later AFIB start.
                if later["label"] in {"AFIB", "N", "AFL"}:
                    boundary = later
                    break
                if later["note"] == ")":
                    boundary = later
                    break

            if boundary is None:
                end = np.nan
                duration = np.nan
                boundary_note = "UNOBSERVED"
                boundary_class = "UNOBSERVED_END"
            else:
                end = boundary["sample"]
                duration = (end - start) / fs
                boundary_note = boundary["note"]

                if boundary_note == ")":
                    boundary_class = "CLOSING_MARKER_RHYTHM_UNCONFIRMED"
                elif boundary["label"] == "N":
                    boundary_class = "FOLLOWED_BY_NSR_MARKER"
                elif boundary["label"] == "AFL":
                    boundary_class = "FOLLOWED_BY_AFL_MARKER"
                elif boundary["label"] == "AFIB":
                    boundary_class = "NEXT_AFIB_START_BEFORE_END"
                else:
                    boundary_class = "OTHER_OR_AMBIGUOUS"

            results.append({
                "patient": patient,
                "record_stem": stem,
                "start_sample": start,
                "start_sec": start / fs,
                "end_sample": end,
                "end_sec": end / fs if pd.notna(end) else np.nan,
                "duration_sec": duration,
                "boundary_annotation": boundary_note,
                "boundary_class": boundary_class,
                "sampling_frequency_hz": fs,
            })

    except Exception as e:
        failures.append({"record_stem": stem, "error": str(e)})

    # Periodically save progress so an interrupted runtime loses less work.
    if i % 25 == 0 or i == len(positive):
        pd.DataFrame(results).to_csv(OUT_CSV, index=False)
        pd.DataFrame(failures).to_csv(FAIL_CSV, index=False)
        print(
            f"Processed {i:,}/{len(positive):,} segments | "
            f"AFib starts parsed: {len(results):,} | "
            f"Failures: {len(failures):,}"
        )

    time.sleep(0.03)

session.close()

# ============================================================
# 3. SUMMARY
# ============================================================
episodes = pd.DataFrame(results)

print("\n========== PARSER SUMMARY ==========")
print(f"Positive segments selected: {len(positive):,}")
print(f"AFib starts parsed: {len(episodes):,}")
print(f"Segments with errors: {len(failures):,}")
print(f"Episode table: {OUT_CSV}")
print(f"Failure log: {FAIL_CSV}")

if not episodes.empty:
    print("\nBoundary classifications:")
    print(episodes["boundary_class"].value_counts(dropna=False))

    print("\nCandidate duration summary (seconds):")
    print(episodes["duration_sec"].describe())

    display(episodes.head(20))


Existing AFib-positive segments to process: 4,588
Processed 25/4,588 segments | AFib starts parsed: 985 | Failures: 0
Processed 50/4,588 segments | AFib starts parsed: 2,386 | Failures: 0
Processed 75/4,588 segments | AFib starts parsed: 3,749 | Failures: 0
Processed 100/4,588 segments | AFib starts parsed: 5,357 | Failures: 0
Processed 125/4,588 segments | AFib starts parsed: 6,486 | Failures: 0
Processed 150/4,588 segments | AFib starts parsed: 7,678 | Failures: 0
Processed 175/4,588 segments | AFib starts parsed: 9,267 | Failures: 0
Processed 200/4,588 segments | AFib starts parsed: 9,897 | Failures: 0
Processed 225/4,588 segments | AFib starts parsed: 10,747 | Failures: 0
Processed 250/4,588 segments | AFib starts parsed: 11,490 | Failures: 0
Processed 275/4,588 segments | AFib starts parsed: 12,429 | Failures: 0
Processed 300/4,588 segments | AFib starts parsed: 13,548 | Failures: 0
Processed 325/4,588 segments | AFib starts parsed: 14,488 | Failures: 0
Processed 350/4,588 segment

,patient,record_stem,start_sample,start_sec,end_sample,end_sec,duration_sec,boundary_annotation,boundary_class,sampling_frequency_hz
0,p00016,p00016_s24,466488,1865.952,537421,2149.684,283.732,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
1,p00016,p00016_s24,547694,2190.776,852851,3411.404,1220.628,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
2,p00016,p00016_s24,853808,3415.232,892551,3570.204,154.972,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
3,p00016,p00016_s24,895749,3582.996,896683,3586.732,3.736,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
4,p00016,p00016_s24,897527,3590.108,901996,3607.984,17.876,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
5,p00016,p00016_s24,903919,3615.676,915370,3661.480,45.804,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
6,p00016,p00016_s24,916037,3664.148,929482,3717.928,53.780,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
7,p00016,p00016_s24,930993,3723.972,938554,3754.216,30.244,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
8,p00016,p00016_s24,940540,3762.160,949028,3796.112,33.952,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0
9,p00016,p00016_s24,950104,3800.416,979571,3918.284,117.868,),CLOSING_MARKER_RHYTHM_UNCONFIRMED,250.0


checking for durations contained in already-checked segments for project data split viability

In [3]:

# ============================================================
# ICENTIA11K: AFib termination feasibility audit
# Uses existing annotation audits only. Downloads no ECG .dat files.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path("/content/drive/MyDrive/icentia11k_afib_audit")
EPISODE_FILE = ROOT / "afib_episode_audit.csv"
SEGMENT_FILE = ROOT / "audited_segments.csv"

assert EPISODE_FILE.exists(), f"Missing: {EPISODE_FILE}"
assert SEGMENT_FILE.exists(), f"Missing: {SEGMENT_FILE}"

episodes = pd.read_csv(EPISODE_FILE)
segments = pd.read_csv(SEGMENT_FILE)

print("Episode audit:", episodes.shape)
print("Segment audit:", segments.shape)
print("Episode columns:", episodes.columns.tolist())

# ---------- Identify the episode start/end columns ----------
def find_col(df, candidates):
    lower = {c.lower().strip(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower:
            return lower[name.lower()]
    for c in df.columns:
        cl = c.lower()
        if any(term in cl for term in candidates):
            return c
    return None

start_col = find_col(episodes, [
    "start_seconds", "start_sec", "start_time_seconds",
    "afib_start_seconds", "start_s", "start_time"
])
end_col = find_col(episodes, [
    "end_seconds", "end_sec", "end_time_seconds",
    "afib_end_seconds", "end_s", "end_time",
    "boundary_seconds", "candidate_end_seconds"
])
duration_col = find_col(episodes, [
    "duration_seconds", "duration_sec", "duration_s",
    "candidate_duration_seconds", "duration"
])
stem_col = find_col(episodes, ["stem", "record", "segment"])
patient_col = find_col(episodes, ["patient_folder", "patient_id", "patient"])

# Episode parser output may use alternate column names.
# Print detected fields so any mismatch is visible.
print("\nDetected fields:")
print(" start:", start_col, "| end:", end_col,
      "| duration:", duration_col, "| stem:", stem_col,
      "| patient:", patient_col)

if duration_col is None:
    raise ValueError(
        "Could not identify episode duration column. "
        "Review the printed column names and set duration_col manually."
    )

# Use explicit interval duration from the existing episode audit.
episodes["duration_s"] = pd.to_numeric(
    episodes[duration_col], errors="coerce"
)

# Derive record stem if needed.
if stem_col is not None:
    episodes["record_stem"] = episodes[stem_col].astype(str)
else:
    seg_col = find_col(episodes, ["segment"])
    if patient_col and seg_col:
        episodes["record_stem"] = (
            episodes[patient_col].astype(str) + "_" +
            episodes[seg_col].astype(str)
        )
    else:
        episodes["record_stem"] = "unknown"

# Extract patient identity from record stem where possible.
def get_patient(stem):
    m = re.search(r"(p\d{5})", str(stem))
    return m.group(1) if m else str(stem)

episodes["patient_id"] = episodes["record_stem"].map(get_patient)

# ---------- Determine eligible windows ----------
# Assumes the candidate AFib interval's closing marker is its end.
# This is a feasibility audit, not yet clinical validation of labels.
episodes = episodes[
    episodes["duration_s"].notna() &
    (episodes["duration_s"] > 0)
].copy()

input_windows = [5, 10]
horizons = [30, 60, 120]
rows = []

for input_s in input_windows:
    for horizon_s in horizons:
        # To observe a termination within the horizon, the AFib interval
        # must last longer than the input, and its end must fall within
        # the next horizon for a positive example.
        #
        # Negative examples need AFib to continue for the entire horizon.
        # For each episode, use one prediction point at its start + input_s.
        eligible = episodes[episodes["duration_s"] > input_s].copy()
        remaining_s = eligible["duration_s"] - input_s

        eligible["remaining_s"] = remaining_s
        eligible["label"] = np.where(
            remaining_s <= horizon_s, 1, 0
        )

        # Positive: interval ends within horizon.
        # Negative: interval continues beyond horizon.
        pos = eligible[eligible["label"] == 1]
        neg = eligible[eligible["label"] == 0]

        rows.append({
            "input_seconds": input_s,
            "horizon_seconds": horizon_s,
            "eligible_episodes": len(eligible),
            "positive_episodes": len(pos),
            "negative_episodes": len(neg),
            "positive_patients": pos["patient_id"].nunique(),
            "negative_patients": neg["patient_id"].nunique(),
            "all_patients": eligible["patient_id"].nunique(),
            "positive_percent": round(100 * eligible["label"].mean(), 2)
                if len(eligible) else np.nan
        })

summary = pd.DataFrame(rows)

print("\n=== FEASIBILITY SUMMARY ===")
display(summary)

# ---------- Duration distribution ----------
print("\n=== CANDIDATE INTERVAL DURATION DISTRIBUTION ===")
display(
    episodes["duration_s"]
    .describe(percentiles=[.10, .25, .50, .75, .90, .95, .99])
    .to_frame("seconds")
)

# ---------- Patient-level support ----------
print("\n=== PATIENT SUPPORT BY HORIZON ===")
patient_rows = []
for horizon_s in horizons:
    for input_s in input_windows:
        eligible = episodes[episodes["duration_s"] > input_s].copy()
        eligible["remaining_s"] = eligible["duration_s"] - input_s
        eligible["label"] = (eligible["remaining_s"] <= horizon_s).astype(int)

        per_patient = eligible.groupby("patient_id")["label"].agg(
            episodes="size", positives="sum"
        )
        patient_rows.append({
            "input_seconds": input_s,
            "horizon_seconds": horizon_s,
            "patients_with_eligible_episodes": len(per_patient),
            "patients_with_positive_examples": int((per_patient["positives"] > 0).sum()),
            "patients_with_negative_examples": int(
                (per_patient["positives"] < per_patient["episodes"]).sum()
            )
        })

display(pd.DataFrame(patient_rows))

# ---------- Save audit results ----------
summary_path = ROOT / "termination_feasibility_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"\nSaved summary to: {summary_path}")
print("\nIMPORTANT:")
print("- This counts one candidate prediction point per annotated interval.")
print("- It does not yet account for waveform quality, recording truncation,")
print("  or whether the annotation boundaries represent true clinical episodes.")
print("- It does not create training labels or download raw ECG.")


Episode audit: (155643, 10)
Segment audit: (118728, 8)
Episode columns: ['patient', 'record_stem', 'start_sample', 'start_sec', 'end_sample', 'end_sec', 'duration_sec', 'boundary_annotation', 'boundary_class', 'sampling_frequency_hz']

Detected fields:
 start: start_sec | end: end_sec | duration: duration_sec | stem: record_stem | patient: patient

=== FEASIBILITY SUMMARY ===


,input_seconds,horizon_seconds,eligible_episodes,positive_episodes,negative_episodes,positive_patients,negative_patients,all_patients,positive_percent
0,5,30,147806,90593,57213,136,138,142,61.29
1,5,60,147806,110765,37041,139,134,142,74.94
2,5,120,147806,125198,22608,141,132,142,84.70
3,10,30,114788,62549,52239,137,137,142,54.49
4,10,60,114788,79753,35035,139,134,142,69.48
5,10,120,114788,92834,21954,141,132,142,80.87



=== CANDIDATE INTERVAL DURATION DISTRIBUTION ===


,seconds
count,155643.000000
mean,99.199700
std,293.241979
min,0.320000
10%,6.068000
25%,9.644000
50%,21.228000
75%,60.494000
90%,197.124000
95%,431.645600



=== PATIENT SUPPORT BY HORIZON ===


,input_seconds,horizon_seconds,patients_with_eligible_episodes,patients_with_positive_examples,patients_with_negative_examples
0,5,30,142,136,138
1,10,30,142,137,137
2,5,60,142,139,134
3,10,60,142,139,134
4,5,120,142,141,132
5,10,120,142,141,132



Saved summary to: /content/drive/MyDrive/icentia11k_afib_audit/termination_feasibility_summary.csv

IMPORTANT:
- This counts one candidate prediction point per annotated interval.
- It does not yet account for waveform quality, recording truncation,
  or whether the annotation boundaries represent true clinical episodes.
- It does not create training labels or download raw ECG.


In [5]:

import os
import numpy as np
import pandas as pd

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/icentia11k_afib_audit"
INPUT_FILE = os.path.join(BASE_DIR, "afib_episode_audit.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "subsampling_audit")
os.makedirs(OUTPUT_DIR, exist_ok=True)

INPUT_SEC = 10
STRIDE_SEC = 10
HORIZON_SEC = 30

SAMPLE_SIZES = [1_000, 2_500, 5_000, 10_000, 20_000]
SEED = 42

# --------------------------------------------------
# LOAD CANDIDATE AFIB INTERVALS
# --------------------------------------------------
episodes = pd.read_csv(INPUT_FILE)

required = {"patient", "record_stem", "start_sec",
            "end_sec", "duration_sec"}
missing = required - set(episodes.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

episodes = episodes.dropna(
    subset=["patient", "record_stem", "start_sec", "end_sec"]
).copy()

episodes = episodes[
    episodes["duration_sec"] > INPUT_SEC
].reset_index(drop=True)

episodes["episode_id"] = np.arange(len(episodes))

print("Candidate intervals:", len(episodes))
print("Patients:", episodes["patient"].nunique())

# --------------------------------------------------
# BUILD WINDOW INDEX ONLY (NO ECG SIGNAL DOWNLOAD)
# Each window represents a prediction opportunity.
# --------------------------------------------------
rng = np.random.default_rng(SEED)
chunks = []

for row in episodes.itertuples(index=False):
    # Window starts at start_sec, then advances by STRIDE_SEC.
    # Keep only full 10-second windows inside the interval.
    starts = np.arange(
        row.start_sec,
        row.end_sec - INPUT_SEC + 1e-9,
        STRIDE_SEC
    )

    window_ends = starts + INPUT_SEC
    remaining = row.end_sec - window_ends

    # Label only windows where the interval endpoint is
    # more than 0 seconds after the window ends.
    valid = remaining > 0
    starts = starts[valid]
    window_ends = window_ends[valid]
    remaining = remaining[valid]

    if len(starts) == 0:
        continue

    chunks.append(pd.DataFrame({
        "patient": row.patient,
        "record_stem": row.record_stem,
        "episode_id": row.episode_id,
        "window_start_sec": starts,
        "window_end_sec": window_ends,
        "remaining_sec": remaining,
        "label": (remaining <= HORIZON_SEC).astype(int)
    }))

windows = pd.concat(chunks, ignore_index=True)
windows["window_id"] = np.arange(len(windows))

print("\nFull eligible window index:", len(windows))
print("Positive rate:", round(windows["label"].mean(), 4))
print("Patients represented:", windows["patient"].nunique())
print("Episodes represented:", windows["episode_id"].nunique())

# --------------------------------------------------
# REPRODUCIBLE SUBSAMPLING
# First sample patients, then windows within patients.
# Patient selection is weighted by available window count,
# so the overall sample approximates uniform window sampling
# while avoiding a purely row-by-row draw.
# --------------------------------------------------
def sample_windows(frame, n, seed):
    if n >= len(frame):
        return frame.copy()

    local_rng = np.random.default_rng(seed)

    patient_counts = frame.groupby("patient").size()
    patients = patient_counts.index.to_numpy()
    probabilities = (
        patient_counts.to_numpy(dtype=float)
        / patient_counts.sum()
    )

    # Allocate a target number to each patient using
    # proportional allocation, with random rounding.
    expected = probabilities * n
    allocation = np.floor(expected).astype(int)
    remainder = n - allocation.sum()

    if remainder > 0:
        fractional = expected - allocation
        eligible = np.flatnonzero(fractional > 0)
        if len(eligible):
            probs = fractional[eligible] / fractional[eligible].sum()
            chosen = local_rng.choice(
                eligible, size=min(remainder, len(eligible)),
                replace=False, p=probs
            )
            allocation[chosen] += 1

    selected = []
    for patient, count in zip(patients, allocation):
        if count == 0:
            continue
        group = frame[frame["patient"] == patient]
        count = min(count, len(group))
        selected.append(
            group.sample(n=count, random_state=seed + int(count))
        )

    result = pd.concat(selected, ignore_index=True)

    # Correct any small rounding discrepancy.
    if len(result) > n:
        result = result.sample(n=n, random_state=seed)
    elif len(result) < n:
        remaining_rows = frame[
            ~frame["window_id"].isin(result["window_id"])
        ]
        extra = remaining_rows.sample(
            n=n - len(result), random_state=seed
        )
        result = pd.concat([result, extra], ignore_index=True)

    return result.reset_index(drop=True)

# --------------------------------------------------
# AUDIT CANDIDATE SAMPLE SIZES
# --------------------------------------------------
summary_rows = []
patient_rows = []
episode_rows = []

for n in SAMPLE_SIZES:
    sampled = sample_windows(windows, min(n, len(windows)), SEED + n)
    tag = f"n_{len(sampled)}"

    sampled.to_csv(
        os.path.join(OUTPUT_DIR, f"{tag}_windows.csv"),
        index=False
    )

    per_patient = sampled.groupby("patient").agg(
        windows=("window_id", "size"),
        positive=("label", "sum"),
        episodes=("episode_id", "nunique")
    ).reset_index()

    per_episode = sampled.groupby(
        ["patient", "episode_id"]
    ).agg(
        windows=("window_id", "size"),
        positive=("label", "sum")
    ).reset_index()

    summary_rows.append({
        "target_n": n,
        "actual_n": len(sampled),
        "patients_covered": sampled["patient"].nunique(),
        "episodes_covered": sampled["episode_id"].nunique(),
        "positive_n": int(sampled["label"].sum()),
        "negative_n": int((sampled["label"] == 0).sum()),
        "positive_rate": sampled["label"].mean(),
        "patients_with_positive": int(
            (per_patient["positive"] > 0).sum()
        ),
        "patients_with_both_labels": int(
            ((per_patient["positive"] > 0) &
             (per_patient["positive"] < per_patient["windows"])).sum()
        ),
        "median_windows_per_patient": per_patient["windows"].median(),
        "max_windows_per_patient": per_patient["windows"].max(),
        "median_episodes_per_patient": per_patient["episodes"].median(),
        "median_windows_per_episode": per_episode["windows"].median(),
        "max_windows_per_episode": per_episode["windows"].max()
    })

    per_patient["sample_size"] = n
    per_episode["sample_size"] = n
    patient_rows.append(per_patient)
    episode_rows.append(per_episode)

summary = pd.DataFrame(summary_rows)
summary.to_csv(
    os.path.join(OUTPUT_DIR, "sample_size_summary.csv"),
    index=False
)

pd.concat(patient_rows).to_csv(
    os.path.join(OUTPUT_DIR, "patient_coverage.csv"),
    index=False
)
pd.concat(episode_rows).to_csv(
    os.path.join(OUTPUT_DIR, "episode_coverage.csv"),
    index=False
)

print("\n=== SAMPLE SIZE COMPARISON ===")
display(summary.round(4))

print("\nSaved audit files to:", OUTPUT_DIR)


Candidate intervals: 114788
Patients: 142

Full eligible window index: 1463212
Positive rate: 0.1766
Patients represented: 142
Episodes represented: 114788

=== SAMPLE SIZE COMPARISON ===


,target_n,actual_n,patients_covered,episodes_covered,positive_n,negative_n,positive_rate,patients_with_positive,patients_with_both_labels,median_windows_per_patient,max_windows_per_patient,median_episodes_per_patient,median_windows_per_episode,max_windows_per_episode
0,1000,1000,113,969,178,822,0.1780,83,81,10.0,14,10.0,1.0,3
1,2500,2500,124,2326,442,2058,0.1768,96,92,25.5,35,24.0,1.0,4
2,5000,5000,129,4297,901,4099,0.1802,109,109,49.0,70,44.0,1.0,6
3,10000,10000,133,7720,1816,8184,0.1816,119,115,97.0,140,77.0,1.0,8
4,20000,20000,134,13122,3559,16441,0.1780,120,118,194.0,280,125.5,1.0,13



Saved audit files to: /content/drive/MyDrive/icentia11k_afib_audit/subsampling_audit


BENEATH THIS IS TESTING

In [6]:

# ============================================================
# ICENTIA11K AFIB INTERVAL-END PREDICTION
# Stage 1: Reproducible patient split and window sampling
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# ---------- Paths ----------
AUDIT_DIR = Path(
    "/content/drive/MyDrive/icentia11k_afib_audit"
)
EPISODE_FILE = AUDIT_DIR / "afib_episode_audit.csv"

OUTPUT_DIR = AUDIT_DIR / "pilot_10s_30s"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Prediction task ----------
INPUT_SECONDS = 10
HORIZON_SECONDS = 30
STRIDE_SECONDS = 10
SAMPLING_RATE_HZ = 250

# ---------- Patient split ----------
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42

# ---------- Pilot sampling controls ----------
# Training: cap the number of selected windows per interval
# and per patient to reduce domination by long intervals
# and high-volume patients.
MAX_WINDOWS_PER_EPISODE = 3
MAX_WINDOWS_PER_PATIENT = 100

# Small natural-distribution samples for initial validation/test
# feature extraction. Increase later if computationally feasible.
VALIDATION_SAMPLE_N = 3000
TEST_SAMPLE_N = 3000

print("Configuration loaded.")
print(f"Input: {INPUT_SECONDS} seconds")
print(f"Prediction horizon: {HORIZON_SECONDS} seconds")
print(f"Stride: {STRIDE_SECONDS} seconds")
print(f"Output folder: {OUTPUT_DIR}")


Configuration loaded.
Input: 10 seconds
Prediction horizon: 30 seconds
Stride: 10 seconds
Output folder: /content/drive/MyDrive/icentia11k_afib_audit/pilot_10s_30s


In [7]:

# Load previously audited candidate AFib intervals.
episodes = pd.read_csv(EPISODE_FILE)

required_columns = {
    "patient", "record_stem", "start_sec",
    "end_sec", "duration_sec"
}

missing = required_columns - set(episodes.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

episodes = episodes.copy()

# Normalize identifiers for consistent grouping.
episodes["patient"] = episodes["patient"].astype(str)
episodes["record_stem"] = episodes["record_stem"].astype(str)

# Remove rows that cannot define a valid interval.
episodes = episodes.dropna(
    subset=["patient", "record_stem", "start_sec", "end_sec"]
)

episodes = episodes[
    (episodes["start_sec"] >= 0) &
    (episodes["end_sec"] > episodes["start_sec"])
].copy()

# Recalculate duration rather than trusting a possibly stale column.
episodes["duration_sec"] = (
    episodes["end_sec"] - episodes["start_sec"]
)

# An interval ID uniquely identifies a row in this audit file.
# It is an analysis key, NOT a model feature.
episodes = episodes.reset_index(drop=True)
episodes["episode_id"] = [
    f"episode_{i:06d}" for i in range(len(episodes))
]

print("Intervals:", len(episodes))
print("Patients:", episodes["patient"].nunique())
print("Records:", episodes["record_stem"].nunique())

display(
    episodes["duration_sec"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
    ).to_frame("duration_seconds")
)


Intervals: 155643
Patients: 142
Records: 4529


,duration_seconds
count,155643.000000
mean,99.199700
std,293.241979
min,0.320000
25%,9.644000
50%,21.228000
75%,60.494000
90%,197.124000
99%,1495.349280
max,4194.288000


In [8]:

# One row per patient. All records and intervals belonging to a
# patient stay in the same partition.
patients = np.array(sorted(episodes["patient"].unique()))

train_patients, remaining_patients = train_test_split(
    patients,
    train_size=TRAIN_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True
)

# Split the remaining patients into validation and test.
test_fraction_of_remaining = (
    TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
)

validation_patients, test_patients = train_test_split(
    remaining_patients,
    test_size=test_fraction_of_remaining,
    random_state=RANDOM_SEED,
    shuffle=True
)

patient_split = {
    **{p: "train" for p in train_patients},
    **{p: "validation" for p in validation_patients},
    **{p: "test" for p in test_patients},
}

episodes["partition"] = episodes["patient"].map(patient_split)

# Verify there is no patient leakage.
assert set(train_patients).isdisjoint(validation_patients)
assert set(train_patients).isdisjoint(test_patients)
assert set(validation_patients).isdisjoint(test_patients)

split_manifest = pd.DataFrame({
    "patient": list(patient_split.keys()),
    "partition": list(patient_split.values())
}).sort_values(["partition", "patient"])

split_manifest.to_csv(
    OUTPUT_DIR / "patient_split_manifest.csv",
    index=False
)

print(split_manifest.groupby("partition")["patient"].nunique())
print("Patient split saved.")


partition
test          22
train         99
validation    21
Name: patient, dtype: int64
Patient split saved.


In [9]:

def make_windows_for_episode(row):
    """
    Create candidate windows for one annotated interval.

    A window ends at prediction_time_sec.
    The 10-second input immediately precedes that time.

    Label:
        1 = interval ends within the next 30 seconds
        0 = interval continues beyond the next 30 seconds

    This predicts an annotated interval boundary, not verified
    clinical conversion to sinus rhythm.
    """
    start = float(row["start_sec"])
    end = float(row["end_sec"])

    # First prediction time after a complete input window.
    first_prediction_time = start + INPUT_SECONDS

    if first_prediction_time >= end:
        return []

    times = np.arange(
        first_prediction_time,
        end,
        STRIDE_SECONDS
    )

    rows = []

    for prediction_time in times:
        seconds_until_end = end - prediction_time

        # Exclude zero-time outcomes: the event is not in the future.
        if seconds_until_end <= 0:
            continue

        # Label based only on time remaining to this interval's end.
        label = int(seconds_until_end <= HORIZON_SECONDS)

        rows.append({
            "patient": row["patient"],
            "record_stem": row["record_stem"],
            "episode_id": row["episode_id"],
            "episode_start_sec": start,
            "episode_end_sec": end,
            "prediction_time_sec": float(prediction_time),
            "input_start_sec": float(
                prediction_time - INPUT_SECONDS
            ),
            "seconds_until_annotated_end": float(seconds_until_end),
            "label": label,
            "partition": row["partition"],
        })

    return rows


# Generate metadata for every eligible candidate window.
window_rows = []

for row in episodes.to_dict("records"):
    window_rows.extend(make_windows_for_episode(row))

windows = pd.DataFrame(window_rows)

if windows.empty:
    raise ValueError("No eligible windows were generated.")

# Stable, unique ID for reproducibility and auditing.
windows = windows.reset_index(drop=True)
windows["window_id"] = [
    f"window_{i:08d}" for i in range(len(windows))
]

print("Eligible windows:", len(windows))
print("Positive rate:", round(windows["label"].mean(), 4))
print("Patients represented:", windows["patient"].nunique())
print("Intervals represented:", windows["episode_id"].nunique())


Eligible windows: 1463212
Positive rate: 0.1766
Patients represented: 142
Intervals represented: 114788


In [10]:

def sample_training_windows(
    train_windows,
    max_per_episode=3,
    max_per_patient=100,
    seed=42
):
    """
    Pilot training sampler.

    1. Shuffle patients.
    2. Shuffle intervals within each patient.
    3. Select up to max_per_episode windows per interval.
    4. Stop at max_per_patient windows per patient.

    Random selection is reproducible for a fixed input table,
    settings, and seed.
    """
    rng = np.random.default_rng(seed)
    selected_parts = []

    for patient in rng.permutation(
        train_windows["patient"].unique()
    ):
        patient_rows = train_windows[
            train_windows["patient"] == patient
        ]

        episode_ids = rng.permutation(
            patient_rows["episode_id"].unique()
        )

        patient_selected = []
        patient_count = 0

        for episode_id in episode_ids:
            episode_rows = patient_rows[
                patient_rows["episode_id"] == episode_id
            ]

            remaining_capacity = (
                max_per_patient - patient_count
            )

            if remaining_capacity <= 0:
                break

            n_to_take = min(
                max_per_episode,
                remaining_capacity,
                len(episode_rows)
            )

            # Uniform sample within this interval.
            chosen_indices = rng.choice(
                episode_rows.index.to_numpy(),
                size=n_to_take,
                replace=False
            )

            chosen = episode_rows.loc[chosen_indices]
            patient_selected.append(chosen)
            patient_count += len(chosen)

        if patient_selected:
            selected_parts.extend(patient_selected)

    if not selected_parts:
        return train_windows.iloc[0:0].copy()

    return (
        pd.concat(selected_parts)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


train_candidates = windows[
    windows["partition"] == "train"
].copy()

train_sample = sample_training_windows(
    train_candidates,
    max_per_episode=MAX_WINDOWS_PER_EPISODE,
    max_per_patient=MAX_WINDOWS_PER_PATIENT,
    seed=RANDOM_SEED
)

print("Training candidate windows:", len(train_candidates))
print("Selected training windows:", len(train_sample))
print("Training positive rate:", round(train_sample["label"].mean(), 4))


Training candidate windows: 1065284
Selected training windows: 8728
Training positive rate: 0.5485


In [11]:

def natural_window_sample(df, target_n, seed):
    """Uniformly sample candidate windows without class balancing."""
    if len(df) <= target_n:
        return df.copy().reset_index(drop=True)

    return df.sample(
        n=target_n,
        replace=False,
        random_state=seed
    ).reset_index(drop=True)


validation_candidates = windows[
    windows["partition"] == "validation"
].copy()

test_candidates = windows[
    windows["partition"] == "test"
].copy()

validation_sample = natural_window_sample(
    validation_candidates,
    VALIDATION_SAMPLE_N,
    RANDOM_SEED + 1
)

test_sample = natural_window_sample(
    test_candidates,
    TEST_SAMPLE_N,
    RANDOM_SEED + 2
)

print("Validation candidates:", len(validation_candidates))
print("Validation sample:", len(validation_sample))
print("Test candidates:", len(test_candidates))
print("Test sample:", len(test_sample))


Validation candidates: 185080
Validation sample: 3000
Test candidates: 212848
Test sample: 3000


In [12]:

sampled = pd.concat(
    [train_sample, validation_sample, test_sample],
    ignore_index=True
)

def summarize_partition(name, df):
    if df.empty:
        return {
            "partition": name,
            "windows": 0,
            "patients": 0,
            "intervals": 0,
            "positive_windows": 0,
            "negative_windows": 0,
            "positive_rate": np.nan,
            "largest_patient_share": np.nan,
            "largest_interval_share": np.nan,
        }

    patient_shares = df["patient"].value_counts(normalize=True)
    interval_shares = df["episode_id"].value_counts(normalize=True)

    return {
        "partition": name,
        "windows": len(df),
        "patients": df["patient"].nunique(),
        "intervals": df["episode_id"].nunique(),
        "positive_windows": int((df["label"] == 1).sum()),
        "negative_windows": int((df["label"] == 0).sum()),
        "positive_rate": df["label"].mean(),
        "largest_patient_share": patient_shares.max(),
        "largest_interval_share": interval_shares.max(),
    }


summary = pd.DataFrame([
    summarize_partition("train", train_sample),
    summarize_partition("validation", validation_sample),
    summarize_partition("test", test_sample),
])

display(summary)

# Inspect patient contribution in training.
train_patient_counts = (
    train_sample.groupby("patient")
    .size()
    .rename("selected_windows")
    .reset_index()
    .sort_values("selected_windows", ascending=False)
)

display(train_patient_counts.head(15))

# Inspect interval contribution in training.
train_episode_counts = (
    train_sample.groupby(["patient", "episode_id"])
    .size()
    .rename("selected_windows")
    .reset_index()
    .sort_values("selected_windows", ascending=False)
)

display(train_episode_counts.head(15))

# Check label availability in every partition.
for name, df in [
    ("train", train_sample),
    ("validation", validation_sample),
    ("test", test_sample)
]:
    print(
        name,
        "labels:",
        df["label"].value_counts().sort_index().to_dict()
    )


,partition,windows,patients,intervals,positive_windows,negative_windows,positive_rate,largest_patient_share,largest_interval_share
0,train,8728,99,3740,4787,3941,0.548465,0.011457,0.000344
1,validation,3000,19,1959,540,2460,0.180000,0.105667,0.003333
2,test,3000,21,2061,611,2389,0.203667,0.101333,0.003333


,patient,selected_windows
0,p00020,100
2,p00060,100
3,p00075,100
31,p00535,100
4,p00084,100
5,p00091,100
6,p00119,100
7,p00177,100
9,p00230,100
10,p00268,100


,patient,episode_id,selected_windows
9,p00020,episode_000165,3
3739,p02015,episode_155634,3
3737,p02015,episode_155499,3
11,p00020,episode_000176,3
3731,p02015,episode_155407,3
3732,p02015,episode_155408,3
3734,p02015,episode_155419,3
2728,p01530,episode_118094,3
2729,p01530,episode_118097,3
2731,p01530,episode_118101,3


train labels: {0: 3941, 1: 4787}
validation labels: {0: 2460, 1: 540}
test labels: {0: 2389, 1: 611}


In [14]:

# ============================================================
# SAMPLING STRATEGY COMPARISON
# Metadata only — no ECG signals are loaded.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

AUDIT_DIR = Path("/content/drive/MyDrive/icentia11k_afib_audit")
OUTPUT_DIR = AUDIT_DIR / "pilot_10s_30s"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_FILE = OUTPUT_DIR / "candidate_window_index.csv"
SPLIT_FILE = OUTPUT_DIR / "patient_split_manifest.csv"

SEED = 42

# Compare strategies at the same approximate training sample size.
TARGET_TRAIN_WINDOWS = 8728

# Adjustable caps for the two interval-balanced strategies.
MAX_PER_EPISODE = 3
MAX_PER_PATIENT = 100

windows = pd.read_csv(WINDOW_FILE)
split_manifest = pd.read_csv(SPLIT_FILE)

required = {
    "patient", "episode_id", "label",
    "prediction_time_sec", "partition"
}
missing = required - set(windows.columns)
if missing:
    raise ValueError(f"Missing columns in window index: {missing}")

# Reuse the existing split manifest as the source of truth.
windows["patient"] = windows["patient"].astype(str)
split_manifest["patient"] = split_manifest["patient"].astype(str)

partition_lookup = dict(zip(
    split_manifest["patient"],
    split_manifest["partition"]
))

windows["partition"] = windows["patient"].map(partition_lookup)

if windows["partition"].isna().any():
    raise ValueError("Some window patients are missing from the split manifest.")

train = windows[windows["partition"] == "train"].copy()

print("Training candidate windows:", len(train))
print("Training patients:", train["patient"].nunique())
print("Training intervals:", train["episode_id"].nunique())
print("Natural training positive rate:", round(train["label"].mean(), 4))


# ------------------------------------------------------------
# Strategy A: natural window sampling
# Each eligible training window has equal selection probability.
# ------------------------------------------------------------
def sample_natural_windows(df, target_n, seed):
    n = min(target_n, len(df))
    return df.sample(n=n, replace=False, random_state=seed).copy()


# ------------------------------------------------------------
# Strategy B/C: interval-first sampling
# Select up to a fixed number of windows per interval.
# Optionally cap each patient's total contribution.
#
# Intervals and patients are visited in random order.
# This is a sampling comparison, not class balancing.
# ------------------------------------------------------------
def sample_interval_first(
    df,
    target_n,
    max_per_episode,
    max_per_patient=None,
    seed=42
):
    rng = np.random.default_rng(seed)
    selected = []
    total_selected = 0

    patient_ids = rng.permutation(df["patient"].unique())

    for patient in patient_ids:
        if total_selected >= target_n:
            break

        patient_df = df[df["patient"] == patient]
        episode_ids = rng.permutation(
            patient_df["episode_id"].unique()
        )

        patient_count = 0

        for episode_id in episode_ids:
            if total_selected >= target_n:
                break

            if (
                max_per_patient is not None
                and patient_count >= max_per_patient
            ):
                break

            episode_df = patient_df[
                patient_df["episode_id"] == episode_id
            ]

            episode_limit = min(
                max_per_episode,
                len(episode_df)
            )

            if max_per_patient is not None:
                remaining_patient_capacity = (
                    max_per_patient - patient_count
                )
                episode_limit = min(
                    episode_limit,
                    remaining_patient_capacity
                )

            remaining_target = target_n - total_selected
            n_take = min(episode_limit, remaining_target)

            if n_take <= 0:
                continue

            chosen_indices = rng.choice(
                episode_df.index.to_numpy(),
                size=n_take,
                replace=False
            )

            selected.append(episode_df.loc[chosen_indices])
            total_selected += n_take
            patient_count += n_take

    if not selected:
        return df.iloc[0:0].copy()

    result = pd.concat(selected, ignore_index=True)

    # Do not silently claim the requested sample size was achieved.
    if len(result) < target_n:
        print(
            f"NOTE: requested {target_n:,} windows; "
            f"caps allowed only {len(result):,}."
        )

    return result


# A. Natural window-weighted sample.
natural = sample_natural_windows(
    train, TARGET_TRAIN_WINDOWS, SEED
)
natural["strategy"] = "natural_windows"

# B. Interval cap, without a patient cap.
equal_interval = sample_interval_first(
    train,
    target_n=TARGET_TRAIN_WINDOWS,
    max_per_episode=MAX_PER_EPISODE,
    max_per_patient=None,
    seed=SEED
)
equal_interval["strategy"] = "equal_interval_cap"

# C. Interval cap plus patient cap.
equal_interval_patient_cap = sample_interval_first(
    train,
    target_n=TARGET_TRAIN_WINDOWS,
    max_per_episode=MAX_PER_EPISODE,
    max_per_patient=MAX_PER_PATIENT,
    seed=SEED
)
equal_interval_patient_cap["strategy"] = "interval_and_patient_caps"


# ------------------------------------------------------------
# Summary metrics
# ------------------------------------------------------------
def summarize_strategy(name, df):
    if df.empty:
        return {
            "strategy": name,
            "windows": 0,
            "patients": 0,
            "intervals": 0,
            "positive_windows": 0,
            "negative_windows": 0,
            "positive_rate": np.nan,
            "largest_patient_share": np.nan,
            "largest_interval_share": np.nan,
            "median_windows_per_patient": np.nan,
            "median_windows_per_interval": np.nan,
        }

    patient_counts = df.groupby("patient").size()
    interval_counts = df.groupby("episode_id").size()

    return {
        "strategy": name,
        "windows": len(df),
        "patients": df["patient"].nunique(),
        "intervals": df["episode_id"].nunique(),
        "positive_windows": int((df["label"] == 1).sum()),
        "negative_windows": int((df["label"] == 0).sum()),
        "positive_rate": df["label"].mean(),
        "largest_patient_share": patient_counts.max() / len(df),
        "largest_interval_share": interval_counts.max() / len(df),
        "median_windows_per_patient": patient_counts.median(),
        "median_windows_per_interval": interval_counts.median(),
    }


comparison = pd.DataFrame([
    summarize_strategy("natural_windows", natural),
    summarize_strategy("equal_interval_cap", equal_interval),
    summarize_strategy(
        "interval_and_patient_caps",
        equal_interval_patient_cap
    ),
])

display(comparison.round(4))

# Save each sample and the comparison for team review.
natural.to_csv(OUTPUT_DIR / "comparison_natural.csv", index=False)
equal_interval.to_csv(
    OUTPUT_DIR / "comparison_equal_interval.csv", index=False
)
equal_interval_patient_cap.to_csv(
    OUTPUT_DIR / "comparison_interval_patient_caps.csv",
    index=False
)
comparison.to_csv(
    OUTPUT_DIR / "sampling_strategy_comparison.csv",
    index=False
)

print("Saved comparison files to:", OUTPUT_DIR)


Training candidate windows: 1065284
Training patients: 99
Training intervals: 80274
Natural training positive rate: 0.1699


,strategy,windows,patients,intervals,positive_windows,negative_windows,positive_rate,largest_patient_share,largest_interval_share,median_windows_per_patient,median_windows_per_interval
0,natural_windows,8728,91,6371,1532,7196,0.1755,0.0206,0.0010,119.0,1.0
1,equal_interval_cap,8728,4,4013,5766,2962,0.6606,0.3734,0.0003,2696.0,3.0
2,interval_and_patient_caps,8728,99,3740,4787,3941,0.5485,0.0115,0.0003,100.0,3.0


Saved comparison files to: /content/drive/MyDrive/icentia11k_afib_audit/pilot_10s_30s


In [13]:

# Save the selected window manifests.
train_sample.to_csv(
    OUTPUT_DIR / "train_windows.csv", index=False
)
validation_sample.to_csv(
    OUTPUT_DIR / "validation_windows.csv", index=False
)
test_sample.to_csv(
    OUTPUT_DIR / "test_windows.csv", index=False
)

# Save the complete candidate-window metadata for reproducibility.
# This contains timestamps and labels, not ECG waveform samples.
windows.to_csv(
    OUTPUT_DIR / "candidate_window_index.csv", index=False
)

summary.to_csv(
    OUTPUT_DIR / "partition_summary.csv", index=False
)

config = {
    "input_seconds": INPUT_SECONDS,
    "prediction_horizon_seconds": HORIZON_SECONDS,
    "stride_seconds": STRIDE_SECONDS,
    "sampling_rate_hz": SAMPLING_RATE_HZ,
    "train_fraction": TRAIN_FRACTION,
    "validation_fraction": VALIDATION_FRACTION,
    "test_fraction": TEST_FRACTION,
    "random_seed": RANDOM_SEED,
    "max_windows_per_episode": MAX_WINDOWS_PER_EPISODE,
    "max_windows_per_patient": MAX_WINDOWS_PER_PATIENT,
    "validation_sample_n": VALIDATION_SAMPLE_N,
    "test_sample_n": TEST_SAMPLE_N,
    "label_definition": (
        "1 if annotated interval ends within (0, 30] seconds; "
        "0 if annotated interval ends more than 30 seconds later"
    ),
    "warning": (
        "Labels indicate annotated interval ends, not verified "
        "clinical AF termination or conversion to sinus rhythm."
    ),
}

with open(OUTPUT_DIR / "configuration.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved pilot artifacts to:", OUTPUT_DIR)
print("Review partition_summary.csv before extracting ECG features.")


Saved pilot artifacts to: /content/drive/MyDrive/icentia11k_afib_audit/pilot_10s_30s
Review partition_summary.csv before extracting ECG features.
